In [1]:
import torch

torch.manual_seed(123)

# Pretend this is your output from Ch2
# [seq_len, d_model] — single sequence, no batch yet
inputs = torch.rand(6, 3)  # 6 tokens, embedding dim 3

# For token 2, compute attention scores against all tokens
query = inputs[1]  # shape [3]

attn_scores = torch.zeros(6)
for i, x in enumerate(inputs):
    attn_scores[i] = torch.dot(query, x)

attn_weights = torch.softmax(attn_scores, dim=0)
context_vec = torch.zeros(3)
for i, x in enumerate(inputs):
    context_vec += attn_weights[i] * x

print("attn_scores:", attn_scores)
print("attn_weights:", attn_weights)  # should sum to 1
print("context_vec:", context_vec)

attn_scores: tensor([0.4602, 1.2304, 0.2611, 1.1189, 0.3408, 1.0023])
attn_weights: tensor([0.1174, 0.2536, 0.0962, 0.2268, 0.1042, 0.2019])
context_vec: tensor([0.4762, 0.2052, 0.6228])


In [2]:
# All attention scores at once: [seq_len, seq_len]
attn_scores = inputs @ inputs.T

# Normalize each row independently
attn_weights = torch.softmax(attn_scores, dim=-1)

# Weighted sum: [seq_len, d_model]
context_vecs = attn_weights @ inputs

print("attn_weights shape:", attn_weights.shape)
print("context_vecs shape:", context_vecs.shape)
print("row 1 matches previous:", torch.allclose(context_vecs[1], context_vec))

attn_weights shape: torch.Size([6, 6])
context_vecs shape: torch.Size([6, 3])
row 1 matches previous: True


In [3]:
torch.manual_seed(123)

d_in = 3   # embedding dim (from inputs)
d_k = 2    # query/key/value dim (hyperparameter)

# Learned projections — no bias, following Raschka
W_q = torch.nn.Parameter(torch.rand(d_in, d_k))
W_k = torch.nn.Parameter(torch.rand(d_in, d_k))
W_v = torch.nn.Parameter(torch.rand(d_in, d_k))

# Project inputs into Q, K, V spaces
Q = inputs @ W_q  # [6, d_k]
K = inputs @ W_k  # [6, d_k]
V = inputs @ W_v  # [6, d_k]

# Scaled dot-product attention
attn_scores = Q @ K.T / d_k**0.5   # [6, 6]
attn_weights = torch.softmax(attn_scores, dim=-1)
context_vecs = attn_weights @ V     # [6, d_k]

print("Q shape:", Q.shape)
print("attn_weights shape:", attn_weights.shape)
print("context_vecs shape:", context_vecs.shape)


Q shape: torch.Size([6, 2])
attn_weights shape: torch.Size([6, 6])
context_vecs shape: torch.Size([6, 2])


In [4]:
import torch
import torch.nn as nn

class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_q = nn.Parameter(torch.rand(d_in, d_out))
        self.W_k = nn.Parameter(torch.rand(d_in, d_out))
        self.W_v = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v

        attn_scores = Q @ K.T / K.shape[-1]**0.5
        attn_weights = torch.softmax(attn_scores, dim=-1)
        return attn_weights @ V

torch.manual_seed(123)
sa = SelfAttentionV1(d_in=3, d_out=2)
print(sa(inputs))

tensor([[0.1671, 0.6265],
        [0.1702, 0.6414],
        [0.1636, 0.6090],
        [0.1705, 0.6434],
        [0.1647, 0.6144],
        [0.1689, 0.6351]], grad_fn=<MmBackward0>)


In [5]:
class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attn_scores = Q @ K.T / K.shape[-1]**0.5
        attn_weights = torch.softmax(attn_scores, dim=-1)
        return attn_weights @ V

torch.manual_seed(123)
sa = SelfAttentionV2(d_in=3, d_out=2)
print(sa(inputs))

tensor([[-0.3799,  0.0564],
        [-0.3938,  0.0638],
        [-0.3806,  0.0574],
        [-0.3902,  0.0617],
        [-0.3810,  0.0575],
        [-0.3906,  0.0623]], grad_fn=<MmBackward0>)


In [6]:
# attention scores from V2
Q = sa.W_q(inputs)
K = sa.W_k(inputs)

attn_scores = Q @ K.T / K.shape[-1]**0.5

# mask: upper triangle = -inf, so softmax makes them 0
mask = torch.triu(torch.ones(6, 6), diagonal=1)
masked_scores = attn_scores.masked_fill(mask.bool(), float('-inf'))
attn_weights = torch.softmax(masked_scores, dim=-1)

print(attn_weights.round(decimals=2))


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4600, 0.5400, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3300, 0.3400, 0.3300, 0.0000, 0.0000, 0.0000],
        [0.2400, 0.2700, 0.2200, 0.2700, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000],
        [0.1600, 0.1800, 0.1500, 0.1800, 0.1500, 0.1700]],
       grad_fn=<RoundBackward1>)


In [7]:
import torch.nn as nn

dropout = nn.Dropout(p=0.5)
attn_weights_dropped = dropout(attn_weights)
print(attn_weights_dropped.round(decimals=2))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6500, 0.0000, 0.0000, 0.0000],
        [0.4800, 0.0000, 0.4500, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.4100, 0.0000, 0.4100, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3500]],
       grad_fn=<RoundBackward1>)


In [8]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        B, T, d_in = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attn_scores = Q @ K.transpose(1, 2) / K.shape[-1]**0.5
        attn_scores = attn_scores.masked_fill(self.mask.bool()[:T, :T], float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ V

# test with a batch
torch.manual_seed(123)
batch = torch.stack([inputs, inputs], dim=0)  # [2, 6, 3]
ca = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
print(ca(batch).shape)

torch.Size([2, 6, 2])
